# Dataset-size scaling plot

Loads the 18 JSON files under `probe_results/`, aggregates TV(true, model) by (architecture, n MDPs),
and plots val TV vs. dataset size with ±std bands.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
results_dir = Path('probe_results')
ns = [10, 30, 50]
sizes = ['small', 'large']
seeds = [0, 1, 2]

# Each entry is a list of per-seed (train_tv, val_tv, emp_tv)
data = {(size, n): [] for size in sizes for n in ns}

for size in sizes:
    for n in ns:
        for seed in seeds:
            path = results_dir / f'{size}_n{n}_seed{seed}.json'
            with open(path) as f:
                summary = json.load(f)['summary']
            data[(size, n)].append((
                summary['mean_tv_true_model_train'],
                summary['mean_tv_true_model_val'],
                summary['mean_tv_true_emp'],
            ))

# Aggregate to mean / std
agg = {}
for key, triples in data.items():
    arr = np.array(triples)  # (3, 3)
    agg[key] = {
        'train_mean': arr[:, 0].mean(), 'train_std': arr[:, 0].std(),
        'val_mean':   arr[:, 1].mean(), 'val_std':   arr[:, 1].std(),
        'emp_mean':   arr[:, 2].mean(),
    }

# Print summary table
print(f"{'n':>4}  {'arch':>5}  {'train (mean±std)':>20}  {'val (mean±std)':>20}")
print('-' * 60)
for n in ns:
    for size in sizes:
        a = agg[(size, n)]
        print(f"{n:>4}  {size:>5}  {a['train_mean']:>10.4f} ± {a['train_std']:.4f}  {a['val_mean']:>10.4f} ± {a['val_std']:.4f}")

emp_baseline = np.mean([agg[(size, n)]['emp_mean'] for size in sizes for n in ns])
print(f"\nEmpirical noise floor (TV true vs counts, n>=30): {emp_baseline:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

colors = {'small': '#1f77b4', 'large': '#d62728'}

for size in sizes:
    train_means = np.array([agg[(size, n)]['train_mean'] for n in ns])
    train_stds  = np.array([agg[(size, n)]['train_std']  for n in ns])
    val_means   = np.array([agg[(size, n)]['val_mean']   for n in ns])
    val_stds    = np.array([agg[(size, n)]['val_std']    for n in ns])

    ax.plot(ns, val_means, '-o', color=colors[size], label=f'{size} (val)', linewidth=2)
    ax.fill_between(ns, val_means - val_stds, val_means + val_stds, color=colors[size], alpha=0.2)

    ax.plot(ns, train_means, '--o', color=colors[size], label=f'{size} (train)', alpha=0.6, linewidth=1.5)
    ax.fill_between(ns, train_means - train_stds, train_means + train_stds, color=colors[size], alpha=0.1)

ax.axhline(emp_baseline, color='gray', linestyle=':', linewidth=1, label=f'empirical floor ≈ {emp_baseline:.2f}')

ax.set_xlabel('Number of MDPs in training family')
ax.set_ylabel('TV distance (model vs. true transition)')
ax.set_title('World-model recovery vs. dataset size')
ax.set_xticks(ns)
ax.set_ylim(bottom=0)
ax.legend(loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('scaling_plot.png', dpi=150, bbox_inches='tight')
plt.show()